In [24]:
from emlp.nn import EMLP#, EMLPBlock, Linear, BiLinear, GatedNonlinearity
from emlp.groups import SO#,O,S,Z, SO13p, SO13, O13
from emlp.reps import vis, V, Scalar
from emlp.datasets import Inertia

import jax.numpy as jnp
import numpy as np
import objax
from tqdm.auto import tqdm
from torch.utils.data import DataLoader
from jax import vmap

import matplotlib.pyplot as plt

In [43]:
class InertiaModel:
    def __init__(self, name) :
        self.name=name
        self.trainset = Inertia(1000) # Initialize dataset with 1000 examples
        self.testset = Inertia(2000)
        self.opt = None
        self.model = None
        self.test_losses = []
        self.train_losses = []
        self.rep_in = None
        self.rep_out = None
        self.testloader = None
        self.trainloder = None

    
    def create_model(self,rep_in, rep_out, group, num_layers=3 ,ch=384):
        self.model = EMLP(rep_in, rep_out, group, num_layers=num_layers, ch=ch)
        self.opt = objax.optimizer.Adam(self.model.vars())
        self.rep_in = rep_in
        self.rep_out = rep_out
        
        
   # @objax.Jit
   # @objax.Function.with_vars(self.model.vars())
    def loss(self,x, y):
        yhat = self.model(x)
        return ((yhat-y)**2).mean()
    
  #  @objax.Jit
  #  @objax.Function.with_vars(self.model.vars()+self.opt.vars())
    def train_op(self,x, y, lr):
        g, v = grad_and_val(x, y)
        self.opt(lr=lr, grads=g)
        return v
    
    def train_model(self,BS=500, lr=3e-3, NUM_EPOCHS=500):
        self.trainloader = DataLoader(self.trainset,batch_size=BS,shuffle=True)
        self.testloader = DataLoader(self.testset,batch_size=BS,shuffle=True)
        
        jit_loss = objax.Jit(self.loss, self.model.vars())
        jit_train = objax.Jit(self.train_op, self.model.vars()+self.opt.vars())
        
        for epoch in tqdm(range(NUM_EPOCHS)):
            self.train_losses.append(np.mean([jit_train(jnp.array(x),jnp.array(y),lr) for (x,y) in self.trainloader]))
            if not epoch%10:
                self.test_losses.append(np.mean([jit_loss(jnp.array(x),jnp.array(y)) for (x,y) in self.testloader]))
    
    def plot(self):
        plt.plot(np.arange(NUM_EPOCHS),self.train_losses,label='Train loss'+name)
        plt.plot(np.arange(0,NUM_EPOCHS,10),self.test_losses,label='Test loss'+name)
        plt.legend()
        plt.yscale('log')
        
    def rel_err(a,b):
        return jnp.sqrt(((a-b)**2).mean())/(jnp.sqrt((a**2).mean())+jnp.sqrt((b**2).mean()))

    def equivariance_err(self,mb):
        x,y = mb
        x,y= jnp.array(x),jnp.array(y)
        gs = G.samples(x.shape[0])
        rho_gin = vmap(self.rep_in.rho_dense)(gs)
        rho_gout = vmap(self.rep_out.rho_dense)(gs)
        y1 = model((rho_gin@x[...,None])[...,0],training=False)
        y2 = (rho_gout@model(x,training=False)[...,None])[...,0]
        return rel_err(y1,y2)
    
    def test_equivariance(self):
        print(f"Average test equivariance error {np.mean([self.equivariance_err(mb) for mb in self.testloader]):.2e}")

In [36]:
G = SO(3)

In [48]:
model1 = InertiaModel('Normal')
rep_in = 5*V+5*Scalar
rep_out = V*V
rep_in = rep_in(G)
rep_out = rep_out(G)

In [49]:
model = EMLP(rep_in, rep_out, G, num_layers=3, ch=384)

NotImplementedError: 